# 从零复现 DDPM：噪声日程、Tiny U-Net 与反向后验

本 Notebook 不使用 `diffusers`、现成 U-Net 或预训练权重。我们用基础 PyTorch 手写 sinusoidal timestep embedding、time-conditioned `ResBlock`、`TinyUNet.forward`、线性 $\beta_t$ schedule、前向加噪 $q(x_t\mid x_0)$、epsilon prediction loss，以及带正确 posterior mean/variance 的逐步反向采样。

重点验证公式和工程边界：同一显式 generator 必须复现噪声与样本；$t=0$ 不得再注入随机噪声；clipping 必须作用于预测的 $x_0$ 并重新计算 posterior mean；训练/validation/test 随机流要分开；制品必须绑定 schedule、数据范围和权重摘要。

全部数据离线合成、固定 seed、CPU 单线程。$8\times8$ 微型图案只验证扩散计算图，不代表现代图像生成质量。

## 1. 两个马尔可夫过程

前向过程逐步加入高斯噪声：

$$q(x_t\mid x_{t-1})=\mathcal N(\sqrt{\alpha_t}x_{t-1},\beta_tI),\quad \alpha_t=1-\beta_t.$$

利用重参数化可以一步得到任意时刻：

$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,\qquad
\bar\alpha_t=\prod_{s=0}^{t}\alpha_s.$$

模型学习 $\epsilon_\theta(x_t,t)$。采样从 $x_T\sim\mathcal N(0,I)$ 开始，按 $t=T-1,\ldots,0$ 迭代近似 $p_\theta(x_{t-1}\mid x_t)$。训练和采样必须使用同一个 schedule 定义。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from copy import deepcopy  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 360728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. 线性 beta schedule 与所有派生系数

教学版使用 32 步线性 $\beta$，范围远大于常见千步 DDPM，是为了让微型采样在 CPU 快速接近纯噪声。改变步数或端点会改变任务，属于模型制品的一部分。

真实后验方差为

$$\tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t},$$

其中定义 $\bar\alpha_{-1}=1$，所以 $\tilde\beta_0=0$。实现不能为了数值方便把 $t=0$ 方差 clamp 成正数后继续采样。

In [ ]:
class DiffusionSchedule(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, timesteps=32, beta_start=1e-4, beta_end=0.18):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if timesteps < 2 or not 0 < beta_start < beta_end < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid diffusion schedule")  # 遇到非法合同立即显式失败。
        betas = torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        alphas = 1.0 - betas  # 计算并保存当前步骤的中间状态。
        alpha_bars = torch.cumprod(alphas, dim=0)  # 计算并保存当前步骤的中间状态。
        alpha_bars_prev = torch.cat([torch.ones(1), alpha_bars[:-1]])  # 计算并保存当前步骤的中间状态。
        posterior_variance = betas * (1 - alpha_bars_prev) / (1 - alpha_bars)  # 计算并保存当前步骤的中间状态。
        posterior_mean_coef1 = betas * alpha_bars_prev.sqrt() / (1 - alpha_bars)  # 计算并保存当前步骤的中间状态。
        posterior_mean_coef2 = (1 - alpha_bars_prev) * alphas.sqrt() / (1 - alpha_bars)  # 计算并保存当前步骤的中间状态。
        self.timesteps = int(timesteps)  # 计算并保存当前步骤的中间状态。
        self.beta_start, self.beta_end = float(beta_start), float(beta_end)  # 计算并保存当前步骤的中间状态。
        for name, tensor in {  # 遍历输入元素以累积或检查结果。
            "betas": betas, "alphas": alphas, "alpha_bars": alpha_bars,  # 执行当前语句以推进本节示例。
            "alpha_bars_prev": alpha_bars_prev,  # 执行当前语句以推进本节示例。
            "posterior_variance": posterior_variance,  # 执行当前语句以推进本节示例。
            "posterior_mean_coef1": posterior_mean_coef1,  # 执行当前语句以推进本节示例。
            "posterior_mean_coef2": posterior_mean_coef2,  # 执行当前语句以推进本节示例。
        }.items():  # 执行当前语句以推进本节示例。
            self.register_buffer(name, tensor)  # 执行当前语句以推进本节示例。

    def forward(self, t):  # 定义本节可复用的核心函数。
        if t.dtype != torch.long or t.ndim != 1:  # 按当前条件选择后续控制路径。
            raise ValueError("t must be int64 [B]")  # 遇到非法合同立即显式失败。
        if (t < 0).any() or (t >= self.timesteps).any():  # 按当前条件选择后续控制路径。
            raise ValueError("timestep out of range")  # 遇到非法合同立即显式失败。
        return self.alpha_bars[t]  # 返回当前分支计算出的结果。

schedule36 = DiffusionSchedule().to(DEVICE)  # 计算并保存当前步骤的中间状态。
assert schedule36.betas.shape == (32,)  # 用受控断言验证关键不变量。
assert torch.all((schedule36.betas > 0) & (schedule36.betas < 1))  # 用受控断言验证关键不变量。
assert torch.all(schedule36.alpha_bars[1:] < schedule36.alpha_bars[:-1])  # 用受控断言验证关键不变量。
assert schedule36.posterior_variance[0].item() == 0.0  # 用受控断言验证关键不变量。
direct_variance_t1 = (schedule36.betas[1] * (1 - schedule36.alpha_bars[0]) /  # 计算并保存当前步骤的中间状态。
                      (1 - schedule36.alpha_bars[1]))  # 执行当前语句以推进本节示例。
assert torch.allclose(schedule36.posterior_variance[1], direct_variance_t1)  # 用受控断言验证关键不变量。

## 3. 正弦 timestep embedding

网络若看不到 $t$，同一个 $x_t$ 无法区分轻噪声和重噪声阶段。对 embedding 维度 $d$，使用不同频率的 sin/cos：

$$e(t)=[\sin(t\omega_0),\ldots,\sin(t\omega_{d/2-1}),
\cos(t\omega_0),\ldots,\cos(t\omega_{d/2-1})].$$

它没有可训练参数，但后接 MLP 让网络学习适合当前任务的时间表示。奇数维会破坏对称拼接，接口直接拒绝。

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, embedding_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if embedding_dim < 4 or embedding_dim % 2:  # 按当前条件选择后续控制路径。
            raise ValueError("time embedding dimension must be even and >= 4")  # 遇到非法合同立即显式失败。
        self.embedding_dim = int(embedding_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, timesteps):  # 定义本节可复用的核心函数。
        if timesteps.ndim != 1:  # 按当前条件选择后续控制路径。
            raise ValueError("timesteps must be [B]")  # 遇到非法合同立即显式失败。
        half = self.embedding_dim // 2  # 计算并保存当前步骤的中间状态。
        frequencies = torch.exp(-math.log(10000.0) *  # 计算并保存当前步骤的中间状态。
                                torch.arange(half, device=timesteps.device) / max(half - 1, 1))  # 计算并保存当前步骤的中间状态。
        angles = timesteps.float()[:, None] * frequencies[None, :]  # 计算并保存当前步骤的中间状态。
        return torch.cat([angles.sin(), angles.cos()], dim=-1)  # 返回当前分支计算出的结果。

time_encoder = SinusoidalTimeEmbedding(16)  # 计算并保存当前步骤的中间状态。
time_probe = time_encoder(torch.tensor([0, 1, 7], dtype=torch.long))  # 计算并保存当前步骤的中间状态。
assert time_probe.shape == (3, 16)  # 用受控断言验证关键不变量。
assert torch.equal(time_probe[0, :8], torch.zeros(8))  # 用受控断言验证关键不变量。
assert torch.equal(time_probe[0, 8:], torch.ones(8))  # 用受控断言验证关键不变量。
assert not torch.equal(time_probe[1], time_probe[2])  # 用受控断言验证关键不变量。

## 4. time-conditioned ResBlock

每个 block 先处理图像特征，再把 time MLP 投影成 `[B,Cout,1,1]` 加入中间激活。输入输出通道不同时，skip path 使用 $1\times1$ projection；相同时使用恒等映射。

GroupNorm 不依赖 batch running statistics，适合扩散训练常见的小 batch。它仍要求 `num_channels` 能被 group 数整除，因此实现显式选择合法 group。

In [ ]:
def valid_groups(channels):  # 定义本节可复用的核心函数。
    return 4 if channels % 4 == 0 else 1  # 返回当前分支计算出的结果。

class TimeConditionedResBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels, time_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels, self.out_channels, self.time_dim = in_channels, out_channels, time_dim  # 计算并保存当前步骤的中间状态。
        self.norm1 = nn.GroupNorm(valid_groups(in_channels), in_channels)  # 计算并保存当前步骤的中间状态。
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)  # 计算并保存当前步骤的中间状态。
        self.time_projection = nn.Linear(time_dim, out_channels)  # 计算并保存当前步骤的中间状态。
        self.norm2 = nn.GroupNorm(valid_groups(out_channels), out_channels)  # 计算并保存当前步骤的中间状态。
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)  # 计算并保存当前步骤的中间状态。
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, time_embedding):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("ResBlock feature shape mismatch")  # 遇到非法合同立即显式失败。
        if time_embedding.shape != (x.shape[0], self.time_dim):  # 按当前条件选择后续控制路径。
            raise ValueError("ResBlock time embedding mismatch")  # 遇到非法合同立即显式失败。
        hidden = self.conv1(F.silu(self.norm1(x)))  # 计算并保存当前步骤的中间状态。
        hidden = hidden + self.time_projection(F.silu(time_embedding))[:, :, None, None]  # 计算并保存当前步骤的中间状态。
        hidden = self.conv2(F.silu(self.norm2(hidden)))  # 计算并保存当前步骤的中间状态。
        return self.skip(x) + hidden  # 返回当前分支计算出的结果。

resblock_probe = TimeConditionedResBlock(4, 8, 16)  # 计算并保存当前步骤的中间状态。
resblock_x = torch.randn(2, 4, 8, 8, requires_grad=True)  # 计算并保存当前步骤的中间状态。
resblock_t = torch.randn(2, 16, requires_grad=True)  # 计算并保存当前步骤的中间状态。
resblock_y = resblock_probe(resblock_x, resblock_t)  # 计算并保存当前步骤的中间状态。
resblock_y.mean().backward()  # 执行当前语句以推进本节示例。
assert resblock_y.shape == (2, 8, 8, 8)  # 用受控断言验证关键不变量。
assert resblock_x.grad is not None and float(resblock_x.grad.norm()) > 0  # 用受控断言验证关键不变量。
assert resblock_t.grad is not None and float(resblock_t.grad.norm()) > 0  # 用受控断言验证关键不变量。

## 5. Tiny U-Net：多尺度路径与 skip concatenation

U-Net 在高分辨率保留局部细节，在低分辨率扩大感受野。教学结构为：input conv → high-resolution ResBlock → stride-2 downsample → middle ResBlock → transposed-conv upsample → 与 high-resolution skip **拼接** → output ResBlock。

拼接使 up block 输入通道为两支通道之和；若误写成相加，结构可能仍能运行，却丢失一半特征表达。输出和输入 shape 必须完全一致，因为目标 epsilon 与 $x_t$ 同形。

In [ ]:
class TinyUNet(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, base_channels=8, time_embedding_dim=16, timesteps=32):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.in_channels, self.timesteps = int(in_channels), int(timesteps)  # 计算并保存当前步骤的中间状态。
        self.base_channels = int(base_channels)  # 计算并保存当前步骤的中间状态。
        self.time_embedding_dim = int(time_embedding_dim)  # 计算并保存当前步骤的中间状态。
        time_dim = 2 * self.time_embedding_dim  # 计算并保存当前步骤的中间状态。
        self.time_encoder = SinusoidalTimeEmbedding(self.time_embedding_dim)  # 计算并保存当前步骤的中间状态。
        self.time_mlp = nn.Sequential(nn.Linear(self.time_embedding_dim, time_dim), nn.SiLU(),  # 计算并保存当前步骤的中间状态。
                                      nn.Linear(time_dim, time_dim))  # 执行当前语句以推进本节示例。
        self.input_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)  # 计算并保存当前步骤的中间状态。
        self.high_block = TimeConditionedResBlock(base_channels, 2 * base_channels, time_dim)  # 计算并保存当前步骤的中间状态。
        self.downsample = nn.Conv2d(2 * base_channels, 2 * base_channels, 4, stride=2, padding=1)  # 计算并保存当前步骤的中间状态。
        self.middle = TimeConditionedResBlock(2 * base_channels, 2 * base_channels, time_dim)  # 计算并保存当前步骤的中间状态。
        self.upsample = nn.ConvTranspose2d(2 * base_channels, 2 * base_channels, 4, stride=2, padding=1)  # 计算并保存当前步骤的中间状态。
        self.up_block = TimeConditionedResBlock(4 * base_channels, base_channels, time_dim)  # 计算并保存当前步骤的中间状态。
        self.output_norm = nn.GroupNorm(valid_groups(base_channels), base_channels)  # 计算并保存当前步骤的中间状态。
        self.output_conv = nn.Conv2d(base_channels, in_channels, 3, padding=1)  # 计算并保存当前步骤的中间状态。

    def forward(self, noisy_images, timesteps):  # 定义本节可复用的核心函数。
        if noisy_images.ndim != 4 or noisy_images.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError("TinyUNet expects configured NCHW input")  # 遇到非法合同立即显式失败。
        if timesteps.shape != (noisy_images.shape[0],) or timesteps.dtype != torch.long:  # 按当前条件选择后续控制路径。
            raise ValueError("timesteps must be int64 [B]")  # 遇到非法合同立即显式失败。
        if (timesteps < 0).any() or (timesteps >= self.timesteps).any():  # 按当前条件选择后续控制路径。
            raise ValueError("timestep out of model range")  # 遇到非法合同立即显式失败。
        time_embedding = self.time_mlp(self.time_encoder(timesteps))  # 计算并保存当前步骤的中间状态。
        high = self.high_block(self.input_conv(noisy_images), time_embedding)  # 计算并保存当前步骤的中间状态。
        middle = self.middle(self.downsample(high), time_embedding)  # 计算并保存当前步骤的中间状态。
        up = self.upsample(middle)  # 计算并保存当前步骤的中间状态。
        if up.shape[-2:] != high.shape[-2:]:  # 按当前条件选择后续控制路径。
            raise ValueError("input spatial dimensions are incompatible with U-Net scale")  # 遇到非法合同立即显式失败。
        fused = torch.cat([up, high], dim=1)  # 计算并保存当前步骤的中间状态。
        return self.output_conv(F.silu(self.output_norm(self.up_block(fused, time_embedding))))  # 返回当前分支计算出的结果。

unet_probe = TinyUNet()  # 计算并保存当前步骤的中间状态。
unet_input = torch.randn(3, 1, 8, 8, requires_grad=True)  # 计算并保存当前步骤的中间状态。
unet_output = unet_probe(unet_input, torch.tensor([0, 7, 31]))  # 计算并保存当前步骤的中间状态。
unet_output.square().mean().backward()  # 执行当前语句以推进本节示例。
assert unet_output.shape == unet_input.shape  # 用受控断言验证关键不变量。
assert unet_input.grad is not None and torch.isfinite(unet_input.grad).all()  # 用受控断言验证关键不变量。
assert float(unet_input.grad.norm()) > 0  # 用受控断言验证关键不变量。
assert unet_probe.time_mlp[0].weight.grad is not None  # 用受控断言验证关键不变量。

## 6. `q_sample`：一次得到任意 $x_t$

训练不必真的循环加噪 $t$ 次。根据闭式公式，从 batch 中为每个样本抽不同 $t$ 和 $\epsilon$，一次构造 $x_t$。随机噪声必须能由调用方传入或通过显式 `torch.Generator` 生成，测试才可重放。

下面用手工给定的 $x_0,t,\epsilon$ 做数值 oracle；同时验证相同 generator seed 逐位相同、不同 seed 不同。全局 `manual_seed` 不是并发服务的请求级随机性合同。

In [ ]:
def extract_coefficient(values, timesteps, reference):  # 定义本节可复用的核心函数。
    if timesteps.dtype != torch.long or timesteps.shape != (reference.shape[0],):  # 按当前条件选择后续控制路径。
        raise ValueError("coefficient timestep shape mismatch")  # 遇到非法合同立即显式失败。
    if (timesteps < 0).any() or (timesteps >= len(values)).any():  # 按当前条件选择后续控制路径。
        raise ValueError("coefficient timestep out of range")  # 遇到非法合同立即显式失败。
    return values[timesteps].reshape(-1, *([1] * (reference.ndim - 1)))  # 返回当前分支计算出的结果。

def q_sample(x0, timesteps, schedule, noise=None, generator=None):  # 定义本节可复用的核心函数。
    if x0.ndim != 4:  # 按当前条件选择后续控制路径。
        raise ValueError("q_sample expects NCHW")  # 遇到非法合同立即显式失败。
    if noise is None:  # 按当前条件选择后续控制路径。
        noise = torch.randn(x0.shape, dtype=x0.dtype, device=x0.device, generator=generator)  # 计算并保存当前步骤的中间状态。
    if noise.shape != x0.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("noise shape mismatch")  # 遇到非法合同立即显式失败。
    alpha_bar = extract_coefficient(schedule.alpha_bars, timesteps, x0)  # 计算并保存当前步骤的中间状态。
    return alpha_bar.sqrt() * x0 + (1 - alpha_bar).sqrt() * noise, noise  # 返回当前分支计算出的结果。

oracle_x0 = torch.tensor([[[[-1., 0.], [.5, 1.]]], [[[.2, -.3], [.4, -.5]]]])  # 计算并保存当前步骤的中间状态。
oracle_t = torch.tensor([0, 7], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
oracle_noise = torch.tensor([[[[.1, -.2], [.3, -.4]]], [[[.5, .6], [-.7, .8]]]])  # 计算并保存当前步骤的中间状态。
oracle_xt, returned_noise = q_sample(oracle_x0, oracle_t, schedule36, noise=oracle_noise)  # 计算并保存当前步骤的中间状态。
oracle_alpha_bar = schedule36.alpha_bars[oracle_t].reshape(2, 1, 1, 1)  # 计算并保存当前步骤的中间状态。
expected_xt = oracle_alpha_bar.sqrt() * oracle_x0 + (1 - oracle_alpha_bar).sqrt() * oracle_noise  # 计算并保存当前步骤的中间状态。
assert torch.allclose(oracle_xt, expected_xt)  # 用受控断言验证关键不变量。
assert torch.equal(returned_noise, oracle_noise)  # 用受控断言验证关键不变量。

generator_a = torch.Generator().manual_seed(991)  # 计算并保存当前步骤的中间状态。
generator_b = torch.Generator().manual_seed(991)  # 计算并保存当前步骤的中间状态。
generator_c = torch.Generator().manual_seed(992)  # 计算并保存当前步骤的中间状态。
sample_a = q_sample(torch.zeros(2, 1, 4, 4), oracle_t, schedule36, generator=generator_a)[0]  # 计算并保存当前步骤的中间状态。
sample_b = q_sample(torch.zeros(2, 1, 4, 4), oracle_t, schedule36, generator=generator_b)[0]  # 计算并保存当前步骤的中间状态。
sample_c = q_sample(torch.zeros(2, 1, 4, 4), oracle_t, schedule36, generator=generator_c)[0]  # 计算并保存当前步骤的中间状态。
assert torch.equal(sample_a, sample_b)  # 用受控断言验证关键不变量。
assert not torch.equal(sample_a, sample_c)  # 用受控断言验证关键不变量。

## 7. epsilon prediction objective

原始 DDPM 常用简化目标

$$L_{simple}=\mathbb E_{x_0,t,\epsilon}\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2.$$

这是对随机 $t$、随机噪声的 Monte Carlo 估计。reduction 必须明确：这里对 batch、通道和空间全部取均值。若误把 target 写成 $x_0$，模型变成另一种 parameterization，反向公式也必须同时改变，不能混搭。

In [ ]:
def epsilon_prediction_loss(model, clean_images, schedule, generator):  # 定义本节可复用的核心函数。
    batch = clean_images.shape[0]  # 计算并保存当前步骤的中间状态。
    timesteps = torch.randint(0, schedule.timesteps, (batch,), generator=generator,  # 计算并保存当前步骤的中间状态。
                              device=clean_images.device)  # 计算并保存当前步骤的中间状态。
    noisy_images, target_noise = q_sample(clean_images, timesteps, schedule, generator=generator)  # 计算并保存当前步骤的中间状态。
    predicted_noise = model(noisy_images, timesteps)  # 计算并保存当前步骤的中间状态。
    if predicted_noise.shape != target_noise.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("epsilon prediction shape mismatch")  # 遇到非法合同立即显式失败。
    return F.mse_loss(predicted_noise, target_noise), timesteps, target_noise  # 返回当前分支计算出的结果。

loss_model_probe = TinyUNet()  # 计算并保存当前步骤的中间状态。
loss_generator = torch.Generator().manual_seed(1701)  # 计算并保存当前步骤的中间状态。
probe_loss, sampled_t, sampled_noise = epsilon_prediction_loss(  # 计算并保存当前步骤的中间状态。
    loss_model_probe, torch.randn(5, 1, 8, 8), schedule36, loss_generator  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
probe_loss.backward()  # 执行当前语句以推进本节示例。
assert probe_loss.ndim == 0 and torch.isfinite(probe_loss)  # 用受控断言验证关键不变量。
assert sampled_t.shape == (5,) and sampled_noise.shape == (5, 1, 8, 8)  # 用受控断言验证关键不变量。
assert loss_model_probe.output_conv.weight.grad is not None  # 用受控断言验证关键不变量。
assert float(loss_model_probe.output_conv.weight.grad.norm()) > 0  # 用受控断言验证关键不变量。

## 8. 受控 $8\times8$ 图案与数据范围

训练分布由竖条、横条、十字三种图案组成，位置和宽度变化，像素严格映射到 `[-1,1]`。扩散模型输出层没有 sigmoid/tanh；采样边界由 $x_0$ clipping 合同控制。

train 与 validation 使用不同 seed 的轻微噪声。真实图像应固定 resize/crop、颜色空间和从 `[0,255]` 到模型范围的映射，并把这些字段放入制品；否则 schedule 相同也不是同一个生成任务。

In [ ]:
def make_diffusion_patterns(count, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    images = -torch.ones(count, 1, 8, 8)  # 计算并保存当前步骤的中间状态。
    for index in range(count):  # 遍历输入元素以累积或检查结果。
        label, position = index % 3, 1 + (index // 3) % 6  # 计算并保存当前步骤的中间状态。
        if label in (0, 2):  # 按当前条件选择后续控制路径。
            images[index, 0, :, position:position + 1] = 1.0  # 计算并保存当前步骤的中间状态。
        if label in (1, 2):  # 按当前条件选择后续控制路径。
            images[index, 0, position:position + 1, :] = 1.0  # 计算并保存当前步骤的中间状态。
    images = (images + 0.03 * torch.randn(images.shape, generator=generator)).clamp(-1, 1)  # 计算并保存当前步骤的中间状态。
    return images  # 返回当前分支计算出的结果。

train36 = make_diffusion_patterns(36, SEED + 1)  # 计算并保存当前步骤的中间状态。
valid36 = make_diffusion_patterns(18, SEED + 2)  # 计算并保存当前步骤的中间状态。
assert train36.shape == (36, 1, 8, 8)  # 用受控断言验证关键不变量。
assert valid36.shape == (18, 1, 8, 8)  # 用受控断言验证关键不变量。
assert float(train36.min()) >= -1 and float(train36.max()) <= 1  # 用受控断言验证关键不变量。
assert not torch.equal(train36[:18], valid36)  # 用受控断言验证关键不变量。

## 9. 受控训练、独立随机流与 validation checkpoint

每步从 train 随机抽 mini-batch，训练 generator 独占自己的随机流。validation 固定一组 $(t,\epsilon)$，因此不同 checkpoint 的 denoising MSE 可公平比较；它不消费训练随机流。test/sample seed 也应另行分配。

第一步检查 time path、卷积路径和输出头梯度。使用 validation 选择 checkpoint，绝不根据最终生成样本手工挑选训练步数。小样本 loss 下降是计算图 smoke test，不是样本质量结论。

In [ ]:
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
model36 = TinyUNet().to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(model36.parameters(), lr=0.004)  # 计算并保存当前步骤的中间状态。
train_generator = torch.Generator().manual_seed(SEED + 100)  # 计算并保存当前步骤的中间状态。
validation_generator = torch.Generator().manual_seed(SEED + 200)  # 计算并保存当前步骤的中间状态。
validation_t = torch.randint(0, schedule36.timesteps, (len(valid36),), generator=validation_generator)  # 计算并保存当前步骤的中间状态。
validation_noise = torch.randn(valid36.shape, generator=validation_generator)  # 计算并保存当前步骤的中间状态。
validation_xt, _ = q_sample(valid36, validation_t, schedule36, noise=validation_noise)  # 计算并保存当前步骤的中间状态。
history, validation_history = [], []  # 计算并保存当前步骤的中间状态。
best_validation, best_state = float("inf"), None  # 计算并保存当前步骤的中间状态。

for step in range(121):  # 遍历输入元素以累积或检查结果。
    indices = torch.randint(0, len(train36), (12,), generator=train_generator)  # 计算并保存当前步骤的中间状态。
    clean_batch = train36[indices]  # 计算并保存当前步骤的中间状态。
    model36.train()  # 执行当前语句以推进本节示例。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    loss, _, _ = epsilon_prediction_loss(model36, clean_batch, schedule36, train_generator)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        gradient_checks = {  # 计算并保存当前步骤的中间状态。
            "input": model36.input_conv.weight.grad.norm(),  # 执行当前语句以推进本节示例。
            "time": model36.time_mlp[0].weight.grad.norm(),  # 执行当前语句以推进本节示例。
            "middle": model36.middle.conv1.weight.grad.norm(),  # 执行当前语句以推进本节示例。
            "output": model36.output_conv.weight.grad.norm(),  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model36.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    history.append(float(loss.detach()))  # 执行当前语句以推进本节示例。
    if step % 10 == 0:  # 按当前条件选择后续控制路径。
        model36.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            validation_mse = float(F.mse_loss(model36(validation_xt, validation_t), validation_noise))  # 计算并保存当前步骤的中间状态。
        validation_history.append((step, validation_mse))  # 执行当前语句以推进本节示例。
        if validation_mse < best_validation:  # 按当前条件选择后续控制路径。
            best_validation, best_state = validation_mse, deepcopy(model36.state_dict())  # 计算并保存当前步骤的中间状态。

assert best_state is not None  # 用受控断言验证关键不变量。
model36.load_state_dict(best_state)  # 执行当前语句以推进本节示例。
assert all(torch.isfinite(value) and float(value) > 0 for value in gradient_checks.values())  # 用受控断言验证关键不变量。
assert sum(history[-20:]) / 20 < sum(history[:20]) / 20  # 用受控断言验证关键不变量。
assert best_validation < float(validation_noise.square().mean())  # 用受控断言验证关键不变量。
print({"train_mse_first20_last20": [sum(history[:20]) / 20, sum(history[-20:]) / 20],  # 执行当前语句以推进本节示例。
       "best_validation_epsilon_mse": best_validation,  # 执行当前语句以推进本节示例。
       "zero_predictor_validation_mse": float(validation_noise.square().mean())})  # 执行当前语句以推进本节示例。

## 10. 正确反向 posterior：先预测并 clip $x_0$

由 epsilon 预测恢复

$$\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\epsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}.$$

若启用 clipping，先把 $\hat x_0$ 截到训练数据范围，再用真实后验均值系数

$$\tilde\mu_t=c_1(t)\hat x_0+c_2(t)x_t$$

重算均值。若只 clip 最终 `x_{t-1}`，就不再对应这个 posterior。$t>0$ 添加方差 $\tilde\beta_t$ 的噪声；$t=0$ 返回均值，不消费噪声。

In [ ]:
def predict_x0_from_epsilon(x_t, timesteps, predicted_noise, schedule, clip=True):  # 定义本节可复用的核心函数。
    alpha_bar = extract_coefficient(schedule.alpha_bars, timesteps, x_t)  # 计算并保存当前步骤的中间状态。
    x0 = (x_t - (1 - alpha_bar).sqrt() * predicted_noise) / alpha_bar.sqrt()  # 计算并保存当前步骤的中间状态。
    return x0.clamp(-1.0, 1.0) if clip else x0  # 返回当前分支计算出的结果。

@torch.no_grad()  # 为下方定义附加声明式配置。
def p_sample(model, x_t, timesteps, schedule, generator, clip_x0=True):  # 定义本节可复用的核心函数。
    if timesteps.shape != (x_t.shape[0],):  # 按当前条件选择后续控制路径。
        raise ValueError("reverse timestep shape mismatch")  # 遇到非法合同立即显式失败。
    predicted_noise = model(x_t, timesteps)  # 计算并保存当前步骤的中间状态。
    predicted_x0 = predict_x0_from_epsilon(x_t, timesteps, predicted_noise, schedule, clip_x0)  # 计算并保存当前步骤的中间状态。
    coef1 = extract_coefficient(schedule.posterior_mean_coef1, timesteps, x_t)  # 计算并保存当前步骤的中间状态。
    coef2 = extract_coefficient(schedule.posterior_mean_coef2, timesteps, x_t)  # 计算并保存当前步骤的中间状态。
    posterior_mean = coef1 * predicted_x0 + coef2 * x_t  # 计算并保存当前步骤的中间状态。
    variance = extract_coefficient(schedule.posterior_variance, timesteps, x_t)  # 计算并保存当前步骤的中间状态。
    active = timesteps > 0  # 计算并保存当前步骤的中间状态。
    noise = torch.zeros_like(x_t)  # 计算并保存当前步骤的中间状态。
    if active.any():  # 按当前条件选择后续控制路径。
        active_shape = (int(active.sum().item()), *x_t.shape[1:])  # 计算并保存当前步骤的中间状态。
        noise[active] = torch.randn(active_shape, dtype=x_t.dtype, device=x_t.device,  # 计算并保存当前步骤的中间状态。
                                    generator=generator)  # 计算并保存当前步骤的中间状态。
    previous = posterior_mean + variance.sqrt() * noise  # 计算并保存当前步骤的中间状态。
    return previous, predicted_x0, posterior_mean  # 返回当前分支计算出的结果。

class ZeroEpsilon(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def forward(self, x, timesteps):  # 定义本节可复用的核心函数。
        return torch.zeros_like(x)  # 返回当前分支计算出的结果。

zero_model = ZeroEpsilon()  # 计算并保存当前步骤的中间状态。
reverse_x = torch.full((2, 1, 2, 2), 0.25)  # 计算并保存当前步骤的中间状态。
reverse_t = torch.tensor([0, 5], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
reverse_generator = torch.Generator().manual_seed(881)  # 计算并保存当前步骤的中间状态。
reverse_result, reverse_x0, reverse_mean = p_sample(  # 计算并保存当前步骤的中间状态。
    zero_model, reverse_x, reverse_t, schedule36, reverse_generator  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
expected_x0 = predict_x0_from_epsilon(reverse_x, reverse_t, torch.zeros_like(reverse_x), schedule36)  # 计算并保存当前步骤的中间状态。
expected_mean = (extract_coefficient(schedule36.posterior_mean_coef1, reverse_t, reverse_x) * expected_x0 +  # 计算并保存当前步骤的中间状态。
                 extract_coefficient(schedule36.posterior_mean_coef2, reverse_t, reverse_x) * reverse_x)  # 执行当前语句以推进本节示例。
replay_noise = torch.zeros_like(reverse_x)  # 计算并保存当前步骤的中间状态。
replay_noise[1:] = torch.randn((1, 1, 2, 2), generator=torch.Generator().manual_seed(881))  # 计算并保存当前步骤的中间状态。
expected_previous = expected_mean + extract_coefficient(  # 计算并保存当前步骤的中间状态。
    schedule36.posterior_variance, reverse_t, reverse_x  # 执行当前语句以推进本节示例。
).sqrt() * replay_noise  # 执行当前语句以推进本节示例。
assert torch.allclose(reverse_x0, expected_x0)  # 用受控断言验证关键不变量。
assert torch.allclose(reverse_mean, expected_mean)  # 用受控断言验证关键不变量。
assert torch.allclose(reverse_result, expected_previous)  # 用受控断言验证关键不变量。
assert torch.equal(reverse_result[0], reverse_mean[0])  # t=0 无随机项
zero_only_generator = torch.Generator().manual_seed(1234)  # 计算并保存当前步骤的中间状态。
generator_state_before = zero_only_generator.get_state().clone()  # 计算并保存当前步骤的中间状态。
_ = p_sample(zero_model, reverse_x[:1], torch.tensor([0]), schedule36, zero_only_generator)  # 计算并保存当前步骤的中间状态。
assert torch.equal(generator_state_before, zero_only_generator.get_state())  # t=0 不消费随机流

large_xt = torch.full((1, 1, 2, 2), 20.0)  # 计算并保存当前步骤的中间状态。
last_t = torch.tensor([31])  # 计算并保存当前步骤的中间状态。
assert predict_x0_from_epsilon(large_xt, last_t, torch.zeros_like(large_xt), schedule36, False).max() > 1  # 用受控断言验证关键不变量。
assert predict_x0_from_epsilon(large_xt, last_t, torch.zeros_like(large_xt), schedule36, True).max() == 1  # 用受控断言验证关键不变量。

## 11. 完整 DDPM 采样与确定性 oracle

采样函数拥有一个请求级 generator：它先生成初始 $x_T$，再按顺序生成每一步 posterior noise。同一 seed、同一模型、同一 schedule 应逐位复现；不同 seed 应产生不同结果。若在循环内部反复把 generator 重置到同一 seed，会让每一步噪声异常相关。

最终 $t=0$ 使用 clipped $\hat x_0$，所以输出应在 `[-1,1]`。这项范围断言只检查接口合同，不评价样本是否像训练数据。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def sample_ddpm(model, schedule, sample_shape, seed):  # 定义本节可复用的核心函数。
    if len(sample_shape) != 4 or sample_shape[1] != model.in_channels:  # 按当前条件选择后续控制路径。
        raise ValueError("sample shape mismatch")  # 遇到非法合同立即显式失败。
    if not 1 <= sample_shape[0] <= 32:  # 按当前条件选择后续控制路径。
        raise ValueError("sample batch out of bounds")  # 遇到非法合同立即显式失败。
    model.eval()  # 执行当前语句以推进本节示例。
    device = next(model.parameters()).device  # 计算并保存当前步骤的中间状态。
    generator = torch.Generator(device=device).manual_seed(int(seed))  # 计算并保存当前步骤的中间状态。
    current = torch.randn(sample_shape, generator=generator, device=device)  # 计算并保存当前步骤的中间状态。
    for timestep in reversed(range(schedule.timesteps)):  # 遍历输入元素以累积或检查结果。
        t = torch.full((sample_shape[0],), timestep, dtype=torch.long, device=device)  # 计算并保存当前步骤的中间状态。
        current, _, _ = p_sample(model, current, t, schedule, generator, clip_x0=True)  # 计算并保存当前步骤的中间状态。
    return current.cpu()  # 返回当前分支计算出的结果。

samples_a = sample_ddpm(model36, schedule36, (4, 1, 8, 8), seed=7001)  # 计算并保存当前步骤的中间状态。
samples_b = sample_ddpm(model36, schedule36, (4, 1, 8, 8), seed=7001)  # 计算并保存当前步骤的中间状态。
samples_c = sample_ddpm(model36, schedule36, (4, 1, 8, 8), seed=7002)  # 计算并保存当前步骤的中间状态。
assert torch.equal(samples_a, samples_b)  # 用受控断言验证关键不变量。
assert not torch.equal(samples_a, samples_c)  # 用受控断言验证关键不变量。
assert samples_a.shape == (4, 1, 8, 8)  # 用受控断言验证关键不变量。
assert float(samples_a.min()) >= -1.0 and float(samples_a.max()) <= 1.0  # 用受控断言验证关键不变量。
assert float((samples_a[0] - samples_a[1]).abs().mean()) > 1e-4  # 用受控断言验证关键不变量。
print({"sample_min_max": [float(samples_a.min()), float(samples_a.max())],  # 执行当前语句以推进本节示例。
       "sample_mean_std": [float(samples_a.mean()), float(samples_a.std())]})  # 执行当前语句以推进本节示例。

## 12. 评估：denoising 指标与生成质量不是一回事

固定 validation $(x_t,t,\epsilon)$ 上的 epsilon MSE 可诊断训练和 checkpoint，但较低 MSE 不必然产生更好的样本。受控例再比较生成图的像素范围与多样性；这些也不能替代分布质量评估。

真实图像常报告 FID/KID、precision/recall、重复记忆与最近邻、条件一致性和人评，并给出置信区间。FID 对样本数和 feature extractor 敏感，不能在几十张微型样本上解释为质量结论。

In [ ]:
model36.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    validation_prediction = model36(validation_xt, validation_t)  # 计算并保存当前步骤的中间状态。
    validation_mse = float(F.mse_loss(validation_prediction, validation_noise))  # 计算并保存当前步骤的中间状态。
    zero_baseline_mse = float(validation_noise.square().mean())  # 计算并保存当前步骤的中间状态。
    reconstructed_x0 = predict_x0_from_epsilon(validation_xt, validation_t,  # 计算并保存当前步骤的中间状态。
                                                validation_prediction, schedule36, clip=True)  # 计算并保存当前步骤的中间状态。

assert math.isclose(validation_mse, best_validation, rel_tol=0, abs_tol=1e-7)  # 用受控断言验证关键不变量。
assert validation_mse < zero_baseline_mse  # 用受控断言验证关键不变量。
assert ((reconstructed_x0 >= -1) & (reconstructed_x0 <= 1)).all()  # 用受控断言验证关键不变量。
pairwise_sample_distance = torch.pdist(samples_a.flatten(1)).mean()  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(pairwise_sample_distance) and float(pairwise_sample_distance) > 0  # 用受控断言验证关键不变量。
print({"fixed_validation_epsilon_mse": validation_mse,  # 执行当前语句以推进本节示例。
       "zero_predictor_mse": zero_baseline_mse,  # 执行当前语句以推进本节示例。
       "generated_pairwise_l2": float(pairwise_sample_distance)})  # 执行当前语句以推进本节示例。

## 13. 制品合同：真实模型配置、schedule、数据快照与外部信任锚

扩散模型的权重不能表达训练时的 beta schedule、epsilon parameterization 或输入映射。builder 必须从真实 `TinyUNet` 与 `DiffusionSchedule` 实例导出 `in_channels/base_channels/time_embedding_dim/timesteps/beta_*`，并先拒绝 model 与 schedule 的步数不一致；硬编码 32 会产生“模型 32、schedule 16 仍能加载”的静默错误。

package 内部 hash 只负责完整性，不是身份认证。这里另外使用发布者侧只读登记 `artifact_id/version -> expected bundle digest`；bundle 对 canonical manifest 与 state 每个 tensor 的 `key/dtype/shape/bytes` 一起做长度分隔哈希。train/validation 图像 split、固定 validation 的 `(t, epsilon)` target、随机流 recipe 和输入 `[-1,1]` 预处理也全部绑定并可重建。整体替换后重签所有内部摘要、伪造 input shape，仍无法改变 package 外的登记值。生产环境应把这一登记落到签名发布元数据或只读制品服务。

In [ ]:
def canonical_json36(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, sort_keys=True, separators=(",", ":"),  # 返回当前分支计算出的结果。
                      ensure_ascii=False).encode("utf-8")  # 计算并保存当前步骤的中间状态。

def _feed_digest36(hasher, payload):  # 定义本节可复用的核心函数。
    hasher.update(len(payload).to_bytes(8, "big"))  # 执行当前语句以推进本节示例。
    hasher.update(payload)  # 执行当前语句以推进本节示例。

def clone_state36(state_dict):  # 定义本节可复用的核心函数。
    if not hasattr(state_dict, "items"):  # 按当前条件选择后续控制路径。
        raise ValueError("state_dict must be a mapping")  # 遇到非法合同立即显式失败。
    cloned = {}  # 计算并保存当前步骤的中间状态。
    for key, tensor in state_dict.items():  # 遍历输入元素以累积或检查结果。
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("state entries must be string -> Tensor")  # 遇到非法合同立即显式失败。
        cloned[key] = tensor.detach().cpu().contiguous().clone()  # 计算并保存当前步骤的中间状态。
    return cloned  # 返回当前分支计算出的结果。

def _update_state_digest36(hasher, state_dict):  # 定义本节可复用的核心函数。
    if not isinstance(state_dict, dict) or not state_dict:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact state_dict must be a non-empty plain dict")  # 遇到非法合同立即显式失败。
    for key in sorted(state_dict):  # 遍历输入元素以累积或检查结果。
        tensor = state_dict[key]  # 计算并保存当前步骤的中间状态。
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid state entry")  # 遇到非法合同立即显式失败。
        cpu = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header = canonical_json36({"key": key, "dtype": str(cpu.dtype),  # 计算并保存当前步骤的中间状态。
                                   "shape": list(cpu.shape)})  # 执行当前语句以推进本节示例。
        raw = cpu.reshape(-1).view(torch.uint8).numpy().tobytes()  # 计算并保存当前步骤的中间状态。
        _feed_digest36(hasher, header)  # 执行当前语句以推进本节示例。
        _feed_digest36(hasher, raw)  # 执行当前语句以推进本节示例。

def canonical_state_digest36(state_dict):  # 定义本节可复用的核心函数。
    hasher = sha256()  # 计算并保存当前步骤的中间状态。
    _feed_digest36(hasher, b"canonical-state-dict-v1")  # 执行当前语句以推进本节示例。
    _update_state_digest36(hasher, state_dict)  # 执行当前语句以推进本节示例。
    return hasher.hexdigest()  # 返回当前分支计算出的结果。

def canonical_bundle_digest36(manifest, state_dict):  # 定义本节可复用的核心函数。
    hasher = sha256()  # 计算并保存当前步骤的中间状态。
    _feed_digest36(hasher, b"canonical-model-bundle-v1")  # 执行当前语句以推进本节示例。
    _feed_digest36(hasher, canonical_json36(manifest))  # 执行当前语句以推进本节示例。
    _update_state_digest36(hasher, state_dict)  # 执行当前语句以推进本节示例。
    return hasher.hexdigest()  # 返回当前分支计算出的结果。

def state_schema36(state_dict):  # 定义本节可复用的核心函数。
    return [{"key": key, "dtype": str(state_dict[key].dtype),  # 返回当前分支计算出的结果。
             "shape": list(state_dict[key].shape)} for key in sorted(state_dict)]  # 执行当前语句以推进本节示例。

def expected_diffusion_data36(timesteps):  # 定义本节可复用的核心函数。
    split_specs = {"train": (36, SEED + 1), "validation": (18, SEED + 2)}  # 计算并保存当前步骤的中间状态。
    splits = {}  # 计算并保存当前步骤的中间状态。
    for name, (count, seed) in split_specs.items():  # 遍历输入元素以累积或检查结果。
        images = make_diffusion_patterns(count, seed)  # 计算并保存当前步骤的中间状态。
        splits[name] = {"count": count, "seed": seed,  # 计算并保存当前步骤的中间状态。
                        "images_sha256": canonical_state_digest36(  # 执行当前语句以推进本节示例。
                            clone_state36({"images": images}))}  # 执行当前语句以推进本节示例。
    validation_generator_replay = torch.Generator().manual_seed(SEED + 200)  # 计算并保存当前步骤的中间状态。
    replay_t = torch.randint(0, timesteps, (18,), generator=validation_generator_replay)  # 计算并保存当前步骤的中间状态。
    replay_noise = torch.randn((18, 1, 8, 8), generator=validation_generator_replay)  # 计算并保存当前步骤的中间状态。
    validation_target_digest = canonical_state_digest36(  # 计算并保存当前步骤的中间状态。
        clone_state36({"timesteps": replay_t, "epsilon_target": replay_noise}))  # 执行当前语句以推进本节示例。
    return {  # 返回当前分支计算出的结果。
        "dataset_recipe": "controlled-8x8-patterns-noise-clamp-v1", "splits": splits,  # 执行当前语句以推进本节示例。
        "training_target": "epsilon", "timestep_sampling": "discrete-uniform-[0,T)",  # 执行当前语句以推进本节示例。
        "noise_distribution": "standard-normal", "train_generator_seed": SEED + 100,  # 执行当前语句以推进本节示例。
        "validation_generator_seed": SEED + 200,  # 执行当前语句以推进本节示例。
        "validation_t_epsilon_sha256": validation_target_digest,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

def schedule_contract36(schedule):  # 定义本节可复用的核心函数。
    return {"kind": "linear", "timesteps": int(schedule.timesteps),  # 返回当前分支计算出的结果。
            "beta_start": float(schedule.beta_start), "beta_end": float(schedule.beta_end),  # 执行当前语句以推进本节示例。
            "betas_digest_sha256": canonical_state_digest36(  # 执行当前语句以推进本节示例。
                clone_state36({"betas": schedule.betas}))}  # 执行当前语句以推进本节示例。

EXPECTED_MODEL_CONFIG36 = {"in_channels": 1, "base_channels": 8,  # 计算并保存当前步骤的中间状态。
                           "time_embedding_dim": 16, "timesteps": 32}  # 执行当前语句以推进本节示例。
EXPECTED_PREPROCESS36 = {"input_shape": [1, 8, 8], "layout": "NCHW",  # 计算并保存当前步骤的中间状态。
                         "dtype": "float32", "data_range": [-1.0, 1.0],  # 执行当前语句以推进本节示例。
                         "recipe": "controlled-pattern-noise-then-clamp-v1"}  # 执行当前语句以推进本节示例。
ARTIFACT_ID36, ARTIFACT_VERSION36 = "vision.controlled-ddpm", "1.0.0"  # 计算并保存当前步骤的中间状态。

def build_diffusion_artifact(model, schedule):  # 定义本节可复用的核心函数。
    if type(model) is not TinyUNet or type(schedule) is not DiffusionSchedule:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher only accepts audited TinyUNet and DiffusionSchedule")  # 遇到非法合同立即显式失败。
    model_config = {"in_channels": model.in_channels, "base_channels": model.base_channels,  # 计算并保存当前步骤的中间状态。
                    "time_embedding_dim": model.time_embedding_dim,  # 执行当前语句以推进本节示例。
                    "timesteps": model.timesteps}  # 执行当前语句以推进本节示例。
    if model_config["timesteps"] != schedule.timesteps:  # 按当前条件选择后续控制路径。
        raise ValueError("model and schedule timesteps must match before publishing")  # 遇到非法合同立即显式失败。
    state = clone_state36(model.state_dict())  # 计算并保存当前步骤的中间状态。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "schema_version": 2, "artifact_id": ARTIFACT_ID36,  # 执行当前语句以推进本节示例。
        "artifact_version": ARTIFACT_VERSION36, "architecture": "TinyUNet",  # 执行当前语句以推进本节示例。
        "model_config": model_config, "model_state_schema": state_schema36(state),  # 执行当前语句以推进本节示例。
        "schedule": schedule_contract36(schedule),  # 执行当前语句以推进本节示例。
        "preprocess": deepcopy(EXPECTED_PREPROCESS36),  # 执行当前语句以推进本节示例。
        "prediction_parameterization": "epsilon",  # 执行当前语句以推进本节示例。
        "data_contract": expected_diffusion_data36(schedule.timesteps),  # 执行当前语句以推进本节示例。
        "state_digest_sha256": canonical_state_digest36(state),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return {"manifest": manifest,  # 返回当前分支计算出的结果。
            "manifest_sha256": sha256(canonical_json36(manifest)).hexdigest(),  # 执行当前语句以推进本节示例。
            "bundle_sha256": canonical_bundle_digest36(manifest, state),  # 执行当前语句以推进本节示例。
            "state_dict": state}  # 执行当前语句以推进本节示例。

def validate_diffusion_contract36(manifest, state_dict):  # 定义本节可复用的核心函数。
    required = {"schema_version", "artifact_id", "artifact_version", "architecture",  # 计算并保存当前步骤的中间状态。
                "model_config", "model_state_schema", "schedule", "preprocess",  # 执行当前语句以推进本节示例。
                "prediction_parameterization", "data_contract", "state_digest_sha256"}  # 执行当前语句以推进本节示例。
    if set(manifest) != required or manifest["schema_version"] != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest schema mismatch")  # 遇到非法合同立即显式失败。
    if (manifest["artifact_id"], manifest["artifact_version"]) != (ARTIFACT_ID36, ARTIFACT_VERSION36):  # 按当前条件选择后续控制路径。
        raise ValueError("artifact identity mismatch")  # 遇到非法合同立即显式失败。
    if manifest["architecture"] != "TinyUNet" or manifest["prediction_parameterization"] != "epsilon":  # 按当前条件选择后续控制路径。
        raise ValueError("unsupported diffusion architecture/parameterization")  # 遇到非法合同立即显式失败。
    model_config, schedule_config = manifest["model_config"], manifest["schedule"]  # 计算并保存当前步骤的中间状态。
    if model_config.get("timesteps") != schedule_config.get("timesteps"):  # 按当前条件选择后续控制路径。
        raise ValueError("model/schedule timestep mismatch")  # 遇到非法合同立即显式失败。
    if model_config != EXPECTED_MODEL_CONFIG36:  # 按当前条件选择后续控制路径。
        raise ValueError("model config mismatch")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"] != EXPECTED_PREPROCESS36:  # 按当前条件选择后续控制路径。
        raise ValueError("input shape/preprocess mismatch")  # 遇到非法合同立即显式失败。
    expected_schedule = schedule_contract36(DiffusionSchedule(32, 1e-4, 0.18))  # 计算并保存当前步骤的中间状态。
    if schedule_config != expected_schedule:  # 按当前条件选择后续控制路径。
        raise ValueError("schedule config or beta tensor mismatch")  # 遇到非法合同立即显式失败。
    if manifest["data_contract"] != expected_diffusion_data36(32):  # 按当前条件选择后续控制路径。
        raise ValueError("train/validation snapshot or randomness recipe mismatch")  # 遇到非法合同立即显式失败。
    expected_schema = state_schema36(TinyUNet(**EXPECTED_MODEL_CONFIG36).state_dict())  # 计算并保存当前步骤的中间状态。
    if manifest["model_state_schema"] != expected_schema or state_schema36(state_dict) != expected_schema:  # 按当前条件选择后续控制路径。
        raise ValueError("model state schema mismatch")  # 遇到非法合同立即显式失败。

def load_trusted_diffusion(artifact):  # 定义本节可复用的核心函数。
    if not isinstance(artifact, dict) or set(artifact) != {  # 按当前条件选择后续控制路径。
            "manifest", "manifest_sha256", "bundle_sha256", "state_dict"}:  # 执行当前语句以推进本节示例。
        raise ValueError("artifact package schema mismatch")  # 遇到非法合同立即显式失败。
    manifest, state = artifact["manifest"], artifact["state_dict"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(manifest, dict):  # 按当前条件选择后续控制路径。
        raise ValueError("manifest must be a dict")  # 遇到非法合同立即显式失败。
    actual_bundle = canonical_bundle_digest36(manifest, state)  # 计算并保存当前步骤的中间状态。
    identity = (manifest.get("artifact_id"), manifest.get("artifact_version"))  # 计算并保存当前步骤的中间状态。
    expected_bundle = PUBLISHER_REGISTRY36.get(identity)  # 计算并保存当前步骤的中间状态。
    if expected_bundle is None or actual_bundle != expected_bundle:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry rejected this bundle")  # 遇到非法合同立即显式失败。
    if artifact["bundle_sha256"] != actual_bundle:  # 按当前条件选择后续控制路径。
        raise ValueError("internal bundle digest mismatch")  # 遇到非法合同立即显式失败。
    if sha256(canonical_json36(manifest)).hexdigest() != artifact["manifest_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest digest mismatch")  # 遇到非法合同立即显式失败。
    if canonical_state_digest36(state) != manifest["state_digest_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("canonical state digest mismatch")  # 遇到非法合同立即显式失败。
    validate_diffusion_contract36(manifest, state)  # 执行当前语句以推进本节示例。
    schedule_config = manifest["schedule"]  # 计算并保存当前步骤的中间状态。
    loaded_schedule = DiffusionSchedule(schedule_config["timesteps"],  # 计算并保存当前步骤的中间状态。
                                        schedule_config["beta_start"], schedule_config["beta_end"])  # 执行当前语句以推进本节示例。
    loaded_model = TinyUNet(**manifest["model_config"])  # 计算并保存当前步骤的中间状态。
    loaded_model.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return loaded_model.eval(), loaded_schedule  # 返回当前分支计算出的结果。

def resign_inside36(artifact):  # 定义本节可复用的核心函数。
    artifact["manifest"]["state_digest_sha256"] = canonical_state_digest36(artifact["state_dict"])  # 计算并保存当前步骤的中间状态。
    artifact["manifest_sha256"] = sha256(canonical_json36(artifact["manifest"])).hexdigest()  # 计算并保存当前步骤的中间状态。
    artifact["bundle_sha256"] = canonical_bundle_digest36(artifact["manifest"], artifact["state_dict"])  # 计算并保存当前步骤的中间状态。
    return artifact  # 返回当前分支计算出的结果。

artifact36 = build_diffusion_artifact(model36, schedule36)  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY36 = MappingProxyType({  # 计算并保存当前步骤的中间状态。
    (ARTIFACT_ID36, ARTIFACT_VERSION36): artifact36["bundle_sha256"]  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。
loaded_model36, loaded_schedule36 = load_trusted_diffusion(artifact36)  # 计算并保存当前步骤的中间状态。
loaded_sample36 = sample_ddpm(loaded_model36, loaded_schedule36, (2, 1, 8, 8), seed=333)  # 计算并保存当前步骤的中间状态。
original_sample36 = sample_ddpm(model36, schedule36, (2, 1, 8, 8), seed=333)  # 计算并保存当前步骤的中间状态。
assert torch.equal(loaded_sample36, original_sample36)  # 用受控断言验证关键不变量。

# builder 必须读取真实实例：16-step 模型与 16-step schedule 的 manifest 都是 16，不再硬编码 32。
short_model36 = TinyUNet(timesteps=16)  # 计算并保存当前步骤的中间状态。
short_schedule36 = DiffusionSchedule(timesteps=16, beta_start=1e-4, beta_end=0.10)  # 计算并保存当前步骤的中间状态。
short_artifact36 = build_diffusion_artifact(short_model36, short_schedule36)  # 计算并保存当前步骤的中间状态。
assert short_artifact36["manifest"]["model_config"]["timesteps"] == 16  # 用受控断言验证关键不变量。
assert short_artifact36["manifest"]["schedule"]["timesteps"] == 16  # 用受控断言验证关键不变量。
assert short_artifact36["manifest"]["schedule"]["beta_end"] == 0.10  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    load_trusted_diffusion(short_artifact36)  # 执行当前语句以推进本节示例。
    raise AssertionError("unpublished self-signed whole replacement must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

# 发布前和加载语义校验都拒绝 model=16 / schedule=32。
try:  # 尝试执行可能失败的受控操作。
    build_diffusion_artifact(short_model36, schedule36)  # 执行当前语句以推进本节示例。
    raise AssertionError("builder must reject model/schedule timestep mismatch")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
mismatch36 = deepcopy(artifact36)  # 计算并保存当前步骤的中间状态。
mismatch36["manifest"]["model_config"]["timesteps"] = 16  # 计算并保存当前步骤的中间状态。
resign_inside36(mismatch36)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    validate_diffusion_contract36(mismatch36["manifest"], mismatch36["state_dict"])  # 执行当前语句以推进本节示例。
    raise AssertionError("semantic validator must reject 16/32 mismatch")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    load_trusted_diffusion(mismatch36)  # 执行当前语句以推进本节示例。
    raise AssertionError("self-signed 16/32 mismatch must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

wrong_shape36 = deepcopy(artifact36)  # 计算并保存当前步骤的中间状态。
wrong_shape36["manifest"]["preprocess"]["input_shape"] = [1, 16, 16]  # 计算并保存当前步骤的中间状态。
resign_inside36(wrong_shape36)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    validate_diffusion_contract36(wrong_shape36["manifest"], wrong_shape36["state_dict"])  # 执行当前语句以推进本节示例。
    raise AssertionError("semantic validator must reject forged input shape")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    load_trusted_diffusion(wrong_shape36)  # 执行当前语句以推进本节示例。
    raise AssertionError("self-signed input shape replacement must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

try:  # 尝试执行可能失败的受控操作。
    PUBLISHER_REGISTRY36[(ARTIFACT_ID36, ARTIFACT_VERSION36)] = short_artifact36["bundle_sha256"]  # 计算并保存当前步骤的中间状态。
    raise AssertionError("publisher registry must be immutable")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 14. 失败模式、复杂度与生产差距

1. **训练预测 epsilon，采样却按 x0 参数化解释**：公式不配套，采样会崩坏。
2. **posterior variance 直接使用 $\beta_t$**：真实 $q(x_{t-1}\mid x_t,x_0)$ 方差是 $\tilde\beta_t$；尤其 $t=0$ 必须为零。
3. **clip 错位置**：应 clip 预测 $x_0$ 后重算 posterior mean，而不是只截断 noisy $x_{t-1}$。
4. **循环内重置 seed**：每步噪声高度相关；一个请求只创建一次 generator 并顺序消费。
5. **训练/validation 共用随机流**：改变评估频率会改变后续训练轨迹；必须拆分 generator。
6. **schedule 或数据 recipe 未由外部发布摘要绑定**：自签内部 hash 仍可整体替换并造成语义错位。

每个 U-Net step 的卷积成本约随 $O(HWC^2)$ 增长，DDPM 总采样成本再乘 $T$。现代生产还需更大的 U-Net/DiT、attention、EMA、mixed precision 数值审计、分布式训练、classifier-free guidance、快速 sampler、内容安全、版权/隐私/记忆审计和目标硬件 p95/p99 压测。

### 论文来源

- Ho, Jain, Abbeel, [*Denoising Diffusion Probabilistic Models*](https://arxiv.org/abs/2006.11239), NeurIPS 2020.
- Sohl-Dickstein et al., [*Deep Unsupervised Learning using Nonequilibrium Thermodynamics*](https://arxiv.org/abs/1503.03585), ICML 2015.
- Ronneberger, Fischer, Brox, [*U-Net: Convolutional Networks for Biomedical Image Segmentation*](https://arxiv.org/abs/1505.04597), MICCAI 2015（多尺度 skip 架构背景）。

本册复现 DDPM 的核心离散公式与微型网络，不声称复现论文数据规模、采样速度或图像质量。